<a href="https://colab.research.google.com/github/ornab74/med-safe-desktop/blob/main/box-breathing-prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Personalized Box Breathing Coach for Colab

This Colab notebook prototypes a meditation app flow with:

- a personalized intake form
- emotion intensity and trigger selection
- adjustable breathing speed
- simulated or manual heart rate input
- Bark voice guidance with more natural pacing
- a live breathing visualizer
- a session summary for future app integration

It uses **Bark** for spoken coaching, based on your original idea, but improves it by:
- preserving sentence order
- adding better pause handling
- using cleaner text normalization
- tailoring script tone to the user profile
- syncing the breath prompts to the chosen pace
- letting you prototype with **simulated heart rate now** and swap in smartwatch data later

In [ ]:
# Install deps for Colab
!pip -q install  sounddevice scipy ipywidgets matplotlib numpy pandas IPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.8 MB/s eta 0:00:00


In [ ]:
# Install custom Bark fork (ebark)
!pip install git+https://github.com/graylan0/bark.git

## 1) Imports and model setup
Run this once. Bark model loading can take a bit.

In [ ]:
import os
import re
import time
import uuid
import math
import queue
import random
import threading
import textwrap
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io.wavfile import write as write_wav
from IPython.display import Audio, display, clear_output, Markdown
import ipywidgets as widgets

from bark import SAMPLE_RATE, generate_audio, preload_models
import warnings
warnings.filterwarnings("ignore")

preload_models()
print("Bark models loaded.")

## 2) Profile and session controls
This is the app-style intake form.

In [ ]:
# Profile widgets
name_w = widgets.Text(description="Name:", placeholder="e.g. Jordan")
location_w = widgets.Text(description="Location:", placeholder="e.g. Brooklyn, NY")
favorite_color_w = widgets.Text(description="Color:", placeholder="e.g. green")
trigger_w = widgets.Dropdown(
    description="Context:",
    options=[
        "After an argument",
        "Feeling overwhelmed",
        "Panic spike",
        "Need to refocus",
        "Before sleep",
        "Custom"
    ],
    value="After an argument"
)
custom_trigger_w = widgets.Text(description="Custom:", placeholder="Describe what's going on")
emotion_w = widgets.IntSlider(description="Emotion:", min=1, max=10, value=6)
energy_w = widgets.IntSlider(description="Energy:", min=1, max=10, value=7)
pace_w = widgets.FloatSlider(
    description="Pace:",
    min=2.0, max=6.0, step=0.5, value=4.0,
    readout_format=".1f"
)
rounds_w = widgets.IntSlider(description="Rounds:", min=2, max=10, value=4)
coach_style_w = widgets.Dropdown(
    description="Coach:",
    options=["Warm", "Grounded", "Gentle", "Direct"],
    value="Warm"
)
speaker_w = widgets.Dropdown(
    description="Voice:",
    options=["v2/en_speaker_6", "v2/en_speaker_3", "v2/en_speaker_9"],
    value="v2/en_speaker_6"
)
hr_mode_w = widgets.ToggleButtons(
    description="Heart rate:",
    options=["Simulated", "Manual"],
    value="Simulated"
)
manual_hr_w = widgets.IntSlider(description="BPM:", min=45, max=140, value=88)
start_btn = widgets.Button(description="Build session", button_style="success")
ui = widgets.VBox([
    widgets.HTML("<h3>Personalized Breathing Coach Intake</h3>"),
    name_w, location_w, favorite_color_w,
    trigger_w, custom_trigger_w,
    emotion_w, energy_w, pace_w, rounds_w,
    coach_style_w, speaker_w, hr_mode_w, manual_hr_w, start_btn
])
display(ui)

## 3) Core logic
This block builds a personalized script, simulates heart rate if needed, and creates a more natural Bark audio response.

### Why this is better than the original snippet
Your original code had a hidden issue: it used `as_completed`, which can scramble sentence order.  
This version keeps sentence order stable, adds text cleanup, smarter pauses, safer chunking, and reusable helpers.

In [ ]:
@dataclass
class SessionProfile:
    name: str
    location: str
    favorite_color: str
    context: str
    emotion_level: int
    energy_level: int
    pace_seconds: float
    rounds: int
    coach_style: str
    speaker_preset: str
    hr_mode: str
    bpm: int

def get_context_value():
    if trigger_w.value == "Custom" and custom_trigger_w.value.strip():
        return custom_trigger_w.value.strip()
    return trigger_w.value

def simulate_hr(emotion_level: int, energy_level: int):
    # Rough prototype logic for a stress-reactive session BPM
    base = 68
    stress_lift = emotion_level * 2.1
    energy_lift = energy_level * 1.4
    noise = random.randint(-3, 5)
    bpm = int(np.clip(base + stress_lift + energy_lift + noise, 52, 125))
    return bpm

def build_profile():
    bpm = manual_hr_w.value if hr_mode_w.value == "Manual" else simulate_hr(emotion_w.value, energy_w.value)
    return SessionProfile(
        name=name_w.value.strip() or "friend",
        location=location_w.value.strip() or "your space",
        favorite_color=favorite_color_w.value.strip() or "blue",
        context=get_context_value(),
        emotion_level=emotion_w.value,
        energy_level=energy_w.value,
        pace_seconds=pace_w.value,
        rounds=rounds_w.value,
        coach_style=coach_style_w.value,
        speaker_preset=speaker_w.value,
        hr_mode=hr_mode_w.value,
        bpm=bpm
    )

def normalized_style(style: str):
    mapping = {
        "Warm": "sound kind, reassuring, and emotionally present",
        "Grounded": "sound calm, steady, and confident",
        "Gentle": "sound soft, patient, and slow",
        "Direct": "sound clear, minimal, and practical",
    }
    return mapping.get(style, "sound calm and supportive")

def state_label(emotion_level: int):
    if emotion_level <= 3:
        return "slightly activated"
    if emotion_level <= 6:
        return "stressed"
    if emotion_level <= 8:
        return "very activated"
    return "highly overwhelmed"

def choose_intro(profile: SessionProfile):
    state = state_label(profile.emotion_level)
    return (
        f"Hey {profile.name}. You're in {profile.location}, and right now you seem {state}. "
        f"If this is happening {profile.context.lower()}, that's okay. "
        f"We're going to do box breathing together at a pace of {profile.pace_seconds:.1f} seconds per side. "
        f"Your current working heart rate is about {profile.bpm} beats per minute."
    )

def choose_affirmation(profile: SessionProfile):
    color_line = f"Picture your favorite color, {profile.favorite_color}, filling the edges of a calm box around you."
    context_line = {
        "After an argument": "You do not have to solve the whole conversation right now. First, come back to your body.",
        "Feeling overwhelmed": "You do not need to carry every thought at once. One breath is enough for this moment.",
        "Panic spike": "Nothing to force. Nothing to outrun. Just this inhale, this hold, this exhale.",
        "Need to refocus": "The next right thing becomes easier when your nervous system slows down.",
        "Before sleep": "Let each exhale soften your body toward rest.",
    }.get(profile.context, "You are allowed to slow down before deciding what comes next.")
    return f"{context_line} {color_line}"

def build_breath_script(profile: SessionProfile):
    style = normalized_style(profile.coach_style)
    intro = choose_intro(profile)
    affirmation = choose_affirmation(profile)

    rounds = []
    for r in range(1, profile.rounds + 1):
        rounds.append(
            f"Round {r}. Inhale for {profile.pace_seconds:.1f}. "
            f"Hold for {profile.pace_seconds:.1f}. "
            f"Exhale for {profile.pace_seconds:.1f}. "
            f"Hold for {profile.pace_seconds:.1f}."
        )

    closing = (
        f"Nice work, {profile.name}. Notice whether your chest, jaw, or shoulders feel even a little softer. "
        f"Your body does not need to be perfect to be safe. "
        f"Take this steadier version of yourself into the next minute."
    )
    return {
        "voice_direction": style,
        "intro": intro,
        "affirmation": affirmation,
        "rounds": rounds,
        "closing": closing
    }

def text_cleanup(text: str):
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace("BPM", "beats per minute")
    text = text.replace(":.1f", "")
    return text

def split_sentences_for_tts(text: str, max_chars: int = 180):
    # Stable sentence segmentation and chunking
    raw = re.split(r"(?<=[.!?])\s+", text_cleanup(text))
    chunks = []
    for sentence in raw:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(sentence) <= max_chars:
            chunks.append(sentence)
        else:
            wrapped = textwrap.wrap(sentence, width=max_chars, break_long_words=False, break_on_hyphens=False)
            chunks.extend(wrapped)
    return chunks

def pause_samples(seconds: float):
    return np.zeros(int(seconds * SAMPLE_RATE), dtype=np.float32)

def envelope(audio: np.ndarray, fade_ms: int = 30):
    if audio.size == 0:
        return audio
    fade_len = int(SAMPLE_RATE * fade_ms / 1000)
    fade_len = min(fade_len, len(audio) // 2)
    if fade_len <= 1:
        return audio
    ramp = np.linspace(0, 1, fade_len)
    audio = audio.astype(np.float32).copy()
    audio[:fade_len] *= ramp
    audio[-fade_len:] *= ramp[::-1]
    return audio

def generate_bark_audio_ordered(text: str, speaker_preset: str = "v2/en_speaker_6"):
    chunks = split_sentences_for_tts(text)
    if not chunks:
        return np.zeros(0, dtype=np.float32)

    outputs = [None] * len(chunks)

    def generate_one(i, sentence):
        try:
            audio = generate_audio(sentence, history_prompt=speaker_preset)
            outputs[i] = envelope(audio)
        except Exception as e:
            print(f"Failed on chunk {i}: {e}")
            outputs[i] = np.zeros(0, dtype=np.float32)

    # Keep order deterministic by storing each future result at its index
    with ThreadPoolExecutor(max_workers=min(2, len(chunks))) as pool:
        futures = [pool.submit(generate_one, i, s) for i, s in enumerate(chunks)]
        for f in futures:
            _ = f.result()

    pieces = []
    for sentence, audio in zip(chunks, outputs):
        if audio is None or audio.size == 0:
            continue
        # Longer pause after reflective lines
        pause_sec = 0.85 if sentence.endswith(("?", "!")) else 0.55
        pieces.extend([audio, pause_samples(pause_sec)])

    if not pieces:
        return np.zeros(0, dtype=np.float32)

    final_audio = np.concatenate(pieces[:-1]).astype(np.float32)
    peak = np.max(np.abs(final_audio)) if final_audio.size else 1.0
    if peak > 0:
        final_audio = final_audio / max(peak, 1e-6) * 0.92
    return final_audio

def save_and_preview_audio(audio: np.ndarray, file_prefix: str = "breathing_coach"):
    file_name = f"{file_prefix}_{uuid.uuid4().hex[:8]}.wav"
    pcm16 = np.int16(np.clip(audio, -1, 1) * 32767)
    write_wav(file_name, SAMPLE_RATE, pcm16)
    display(Audio(file_name))
    print(f"Saved: {file_name}")
    return file_name

## 4) Emotion heatmap and breath visualizer
A simple emotional map plus a live box-breathing animation.

In [ ]:
def plot_emotion_heatmap(emotion_level: int, energy_level: int):
    grid = np.zeros((10, 10))
    for i in range(10):
        for j in range(10):
            grid[i, j] = math.exp(-(((i + 1 - energy_level) ** 2 + (j + 1 - emotion_level) ** 2) / 8.0))

    plt.figure(figsize=(6, 5))
    plt.imshow(grid, origin="lower", aspect="auto")
    plt.colorbar(label="Activation")
    plt.xticks(range(10), range(1, 11))
    plt.yticks(range(10), range(1, 11))
    plt.xlabel("Energy / Restlessness")
    plt.ylabel("Emotion Intensity")
    plt.title("Emotional Activation Heatmap")
    plt.show()

def draw_breath_box(phase: str, progress: float, color_name: str):
    plt.figure(figsize=(4, 4))
    xs = [0, 1, 1, 0, 0]
    ys = [0, 0, 1, 1, 0]
    plt.plot(xs, ys, linewidth=4)
    plt.scatter([progress], [0], s=120) if phase == "inhale" else None
    plt.scatter([1], [progress], s=120) if phase == "hold_top" else None
    plt.scatter([1-progress], [1], s=120) if phase == "exhale" else None
    plt.scatter([0], [1-progress], s=120) if phase == "hold_bottom" else None
    plt.xlim(-0.1, 1.1); plt.ylim(-0.1, 1.1)
    plt.xticks([]); plt.yticks([])
    plt.title(f"{phase.replace('_', ' ').title()} • imagined color: {color_name}")
    plt.show()

def run_box_visualizer(profile: SessionProfile):
    phase_names = ["inhale", "hold_top", "exhale", "hold_bottom"]
    step_count = 6
    for round_idx in range(profile.rounds):
        clear_output(wait=True)
        display(Markdown(f"### Live Box Breathing Visualizer\nRound {round_idx + 1} of {profile.rounds}"))
        for phase in phase_names:
            for step in range(step_count + 1):
                clear_output(wait=True)
                display(Markdown(f"### Live Box Breathing Visualizer\nRound {round_idx + 1} of {profile.rounds}"))
                draw_breath_box(phase, step / step_count, profile.favorite_color)
                time.sleep(profile.pace_seconds / step_count)

## 5) Build the session
This creates:
- the profile
- the personalized script
- the heatmap
- the Bark voice file
- a session summary table

In [ ]:

session_out = widgets.Output()
display(session_out)

def on_start_clicked(_):
    with session_out:
        clear_output(wait=True)
        profile = build_profile()
        script = build_breath_script(profile)

        print("Session profile:")
        print(asdict(profile))
        print()

        display(Markdown("## Personalized Coach Script"))
        full_script = "\n\n".join([
            script["intro"],
            script["affirmation"],
            *script["rounds"],
            script["closing"]
        ])
        display(Markdown(full_script))

        print("\nRendering emotion heatmap...")
        plot_emotion_heatmap(profile.emotion_level, profile.energy_level)

        print("Generating Bark audio...")
        audio = generate_bark_audio_ordered(full_script, speaker_preset=profile.speaker_preset)
        wav_file = save_and_preview_audio(audio)

        summary_df = pd.DataFrame([asdict(profile)])
        display(Markdown("## Session Summary"))
        display(summary_df)

        display(Markdown(
            f"**Prototype note:** HR mode is **{profile.hr_mode}**. "
            f"Right now this notebook supports manual or simulated BPM. "
            f"That can later be swapped for Apple Watch or HealthKit input in Swift."
        ))

start_btn.on_click(on_start_clicked)
print("Click 'Build session' above.")

## 6) Optional: run the visualizer after generating the session
This gives a simple guided on-screen box animation.

In [ ]:
run_visual_btn = widgets.Button(description="Run visualizer", button_style="info")
visual_out = widgets.Output()
display(run_visual_btn, visual_out)

def on_run_visual(_):
    with visual_out:
        clear_output(wait=True)
        profile = build_profile()
        run_box_visualizer(profile)

run_visual_btn.on_click(on_run_visual)

## 7) Product ideas to evolve this prototype

### Immediate upgrades
- sentiment-aware script branching
- post-argument mode vs panic mode vs sleep mode
- optional ambient pad underneath the voice
- dynamic pace adjustment if heart rate trends down
- short debrief after session: “Do you feel better, same, or worse?”

### App architecture ideas
- **Swift frontend** for watch + phone input
- **Python voice service** for custom guided content
- health data adapter for real BPM
- session memory with privacy-safe journaling
- adaptive scripts based on recent triggers and response patterns

### Important note
This is a wellness prototype, not a medical device. If a user reports chest pain, fainting, severe distress, or persistent panic symptoms, route them away from the meditation flow and toward urgent support.